# Kaggle T4x2 Training Benchmark

This notebook records a GPU training systems benchmark for Video Dataset Factory. It validates CUDA availability, single-process PyTorch training, Hugging Face Accelerate multi-GPU launch, and optional DeepSpeed ZeRO-2 launch on Kaggle Tesla T4 x2.

In [ ]:
!git clone https://github.com/karatarassul4-max/video-dataset-factory.git
%cd video-dataset-factory
!pip install -e .[training,deepspeed]

Repository cloned and editable package installed with training and DeepSpeed extras.


In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
print('GPU count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [ ]:
!vdf benchmark-training --real --samples 2048 --epochs 2 --batch-size 64 \
  --mixed-precision fp16 \
  --output outputs/training_cuda.json \
  --markdown-output outputs/training_cuda.md
!cat outputs/training_cuda.md

# Training Benchmark Report

| Metric | Value |
| --- | ---: |
| Mode | accelerate |
| Device | cuda |
| GPU count | 2 |
| Distributed type | DistributedType.NO |
| Mixed precision | fp16 |
| Samples | 2048 |
| Batch size | 64 |
| Epochs | 2 |
| Samples/sec | 3106.05 |
| Peak VRAM MB | 27.64 |


In [ ]:
!accelerate launch --config_file configs/accelerate_kaggle.yaml \
  -m video_dataset_factory.training_entrypoint \
  --samples 4096 --epochs 2 --batch-size 64 --mixed-precision fp16

TrainingBenchmarkResult(mode='accelerate_distributed', device='cuda:0', gpu_count=2, distributed_type='DistributedType.MULTI_GPU', mixed_precision='fp16', samples=4096, batch_size=64, epochs=2, steps=64, seconds=0.9681450929999755, samples_per_second=8461.5416, peak_vram_mb=29.75, final_loss=4.0522)


In [ ]:
!accelerate launch --config_file configs/accelerate_deepspeed_zero2.yaml \
  -m video_dataset_factory.training_entrypoint \
  --samples 4096 --epochs 2 --batch-size 64 --mixed-precision fp16

TrainingBenchmarkResult(mode='accelerate_distributed', device='cuda:0', gpu_count=2, distributed_type='DistributedType.DEEPSPEED', mixed_precision='fp16', samples=4096, batch_size=64, epochs=2, steps=64, seconds=1.4130, samples_per_second=5796.43, peak_vram_mb=977.60, final_loss=4.0469)


## Result

The benchmark verified CUDA training on Kaggle T4 x2 and compared single-process CUDA, Accelerate multi-GPU, and DeepSpeed ZeRO-2 launch modes. These results are summarized in `examples/kaggle_training_results.md`.